# Chapter 7 · QAOA — Quantum Approximate Optimization Algorithm

## Objectives

1. Understand QAOA as a variational algorithm for combinatorial optimization.
2. Implement QAOA for the MaxCUT problem on a small graph.
3. Analyze solution quality as a function of the number of layers $p$.

---

## 7B.1 The MaxCUT Problem

Given a graph $G=(V,E)$, MaxCUT seeks a bipartition $(S, \bar{S})$ of the vertices that maximizes the number of edges between $S$ and $\bar{S}$.

The Cost Hamiltonian is:

$$H_C = \sum_{(i,j) \in E} w_{ij} \frac{I - Z_i Z_j}{2}$$

and the QAOA ansatz of depth $p$ is:

$$|\boldsymbol{\gamma}, \boldsymbol{\beta}\rangle = e^{-i\beta_p H_B} e^{-i\gamma_p H_C} \cdots e^{-i\beta_1 H_B} e^{-i\gamma_1 H_C} |s\rangle$$

In [ ]:
import sys, os
sys.path.insert(0, os.path.join(os.getcwd(), '..', '..'))

import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import minimize
from qiskit import QuantumCircuit
from qiskit.quantum_info import SparsePauliOp, Statevector
from qiskit_aer import AerSimulator
from src.visualization import QuantumVisualization

print('Modules loaded.')

In [ ]:
# 4-node graph (pedagogical example)
n_nodes = 4
edges = [(0, 1, 1.0), (0, 2, 1.0), (1, 3, 1.0), (2, 3, 1.0)]  # (u, v, weight)

print('MaxCUT Graph:')
for u, v, w in edges:
    print(f'  Edge ({u},{v}) weight={w}')

# Cost Hamiltonian
def maxcut_hamiltonian(n: int, edges: list) -> SparsePauliOp:
    """Builds H_C for MaxCUT."""
    ops = []
    for u, v, w in edges:
        # (I - Z_u Z_v) / 2  ×  weight
        I_term = 'I' * n
        ZZ_term = list('I' * n)
        ZZ_term[n - 1 - u] = 'Z'
        ZZ_term[n - 1 - v] = 'Z'
        ops.append((''.join(I_term), w / 2))
        ops.append((''.join(ZZ_term), -w / 2))
    return SparsePauliOp.from_list(ops).simplify()

H_cost = maxcut_hamiltonian(n_nodes, edges)
print('\nCost Hamiltonian:')
print(H_cost)

In [ ]:
def qaoa_circuit(n: int, edges: list, gamma: list, beta: list) -> QuantumCircuit:
    """Builds the QAOA circuit of depth p.

    Parameters
    ----------
    n : int
        Number of qubits (= number of nodes).
    edges : list
        List of edges (u, v, w).
    gamma : list
        Angles for the cost operator (length p).
    beta : list
        Angles for the mixing operator (length p).
    """
    p = len(gamma)
    qc = QuantumCircuit(n)

    # Initial state: uniform superposition
    qc.h(range(n))

    for layer in range(p):
        # Problem operator (ZZ)
        for u, v, w in edges:
            qc.cx(u, v)
            qc.rz(2 * gamma[layer] * w, v)
            qc.cx(u, v)

        # Mixing operator (X)
        for i in range(n):
            qc.rx(2 * beta[layer], i)

    return qc


def qaoa_cost(params: np.ndarray, n: int, edges: list, p: int, H_matrix) -> float:
    """Evaluates the expected value of H_C for the given parameters."""
    gamma = params[:p]
    beta  = params[p:]
    qc = qaoa_circuit(n, edges, gamma, beta)
    sv = Statevector(qc)
    return float(-np.real(sv.data.conj() @ H_matrix @ sv.data))  # negative for min


# Optimization for p=1
p = 1
H_matrix = H_cost.to_matrix()

energy_history = []

def tracked_cost(params):
    val = qaoa_cost(params, n_nodes, edges, p, H_matrix)
    energy_history.append(-val)
    return val

np.random.seed(7)
params0 = np.random.uniform(0, np.pi, 2 * p)

result = minimize(tracked_cost, params0, method='COBYLA',
                  options={'maxiter': 300})

best_cut = -result.fun
print(f'QAOA p={p}:')
print(f'  Approximate cut = {best_cut:.4f}')
print(f'  Exact maximum cut = {len(edges)}')
print(f'  Approximation ratio = {best_cut / len(edges):.4f}')

In [ ]:
# Visualization
fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(energy_history, color='#58a6ff', linewidth=1.5)
ax.axhline(len(edges), color='#f78166', linestyle='--',
           label=f'Classical optimum = {len(edges)}')
ax.set_xlabel('Iteration')
ax.set_ylabel('Cut value ⟨H_C⟩')
ax.set_title(f'QAOA p={p} convergence — MaxCUT 4 nodes')
ax.legend()
ax.grid(alpha=0.3)
ax.set_facecolor('#161b22')
fig.patch.set_facecolor('#0d1117')
plt.tight_layout()
plt.show()

## 7B.2 Proposed Exercises

1. Repeat QAOA for $p = 2$ and $p = 3$. Does solution quality improve with $p$?

2. Apply QAOA to the 3-coloring problem of a graph expressed as a penalty Hamiltonian.

3. Implement the Balanced Partition Problem and solve it with QAOA.